# Stock Analysis
## Greedy Prediction

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Custom color palette - vibrant and modern
custom_palette = ['#9500ff', '#ff0059', '#ff8c00', '#b4e600', '#0fffdb']

# Set elegant minimalist theme with clean sans-serif fonts
sns.set_theme(
    context='paper',
    style='whitegrid',
    palette=custom_palette,
    rc={
        'font.family': 'sans-serif',
        'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
        'axes.labelcolor': '#2962ff',
        'axes.edgecolor': '#e0e0e0',
        'axes.linewidth': 0.8,
        'grid.color': '#f5f5f5',
        'grid.linewidth': 0.5,
        'xtick.color': '#666666',
        'ytick.color': '#666666',
        'text.color': '#333333',
        'axes.titlesize': 14,
        'axes.titleweight': 'bold',
        'axes.labelsize': 11,
        'axes.labelweight': 'bold',
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'font.size': 10
    }
)

In [0]:
%sql --quiet
select
  *
from
  stockanalysis.stocks;

In [0]:
stocks_df = _sqldf.toPandas()

In [0]:
def calc_volatility(stock_prices: list[np.float64], weights: list[np.float64]):

    if len(stock_prices) != len(weights):
        raise ValueError("Stock prices and weights must have the same length.")

    weights_sum = np.sum(weights)
    
    weighted_mean = np.sum(stock_prices * weights) / weights_sum
    volatility = 0

    for price, weight in zip(stock_prices, weights):
        volatility += weight * np.square(price - weighted_mean)
    
    volatility = np.sqrt(volatility / weights_sum)
    
    return volatility, weighted_mean


In [0]:
def monte_carlo_simulation(ticker, n_simulations=1000, forecast_days=30, look_back=90, alpha=1, verbose=True):
    """
    Run Monte Carlo simulation for stock price prediction (simulation only, no plotting)
    
    Parameters:
    -----------
    ticker : str
        Stock ticker symbol
    n_simulations : int
        Number of simulation iterations
    forecast_days : int
        Number of days to forecast
    look_back : int
        Historical window size for analysis
    alpha : float
        Weight parameter for time decay
    verbose : bool
        Whether to print summary statistics
    
    Returns:
    --------
    dict : Simulation results with all statistics and metadata
    """
    # Prepare data
    ticker_df = stocks_df[stocks_df['Ticker'] == ticker].copy()
    ticker_df = ticker_df.sort_values(by='Date').reset_index(drop=True)
    ticker_df['Date'] = pd.to_datetime(ticker_df['Date'])
    
    # Filter for last 3 years
    last_date = ticker_df['Date'].max()
    three_years_ago = last_date - pd.DateOffset(years=3)
    ticker_df = ticker_df[ticker_df['Date'] >= three_years_ago].reset_index(drop=True)
    
    dataset = ticker_df["Close"].values
    dates = ticker_df["Date"]
    
    # Store all simulations
    all_predictions = np.zeros((n_simulations, forecast_days))
    
    # Run simulations
    for sim in range(n_simulations):
        window = dataset[-look_back:].copy()
        predicted = []
        positives = 0
        negatives = 0
        total = 0
        
        # Weight array for time decay
        w = np.array([i / look_back for i in range(look_back)])
        w = [ np.pow(x, alpha) for x in w]
        
        for i in range(forecast_days):
            after = np.insert(window[:-1], 0, 0)
            diff = after - window
            
            for (j, x) in enumerate(diff):
                total += w[j] * abs(x)
                if x > 0:
                    positives += w[j] * abs(x)
                else:
                    negatives += w[j] * abs(x)
            
            prob = positives / total if total != 0 else 0.5
            vol, weighted_mean = calc_volatility(diff, w)
            
            new_diff = np.random.normal(loc=weighted_mean, scale=vol)
            
            if np.random.rand() < prob:
                new_diff *= -1
            
            forecast = max(window[-1] + new_diff, 0)
            predicted.append(forecast)
            window = np.append(window[1:], forecast)
        
        all_predictions[sim, :] = predicted
    
    # Calculate statistics
    mean_prediction = np.mean(all_predictions, axis=0)
    median_prediction = np.median(all_predictions, axis=0)
    std_prediction = np.std(all_predictions, axis=0)
    
    # Calculate confidence intervals
    percentile_95_upper = np.percentile(all_predictions, 97.5, axis=0)
    percentile_95_lower = np.percentile(all_predictions, 2.5, axis=0)
    percentile_80_upper = np.percentile(all_predictions, 90, axis=0)
    percentile_80_lower = np.percentile(all_predictions, 10, axis=0)
    percentile_50_upper = np.percentile(all_predictions, 75, axis=0)
    percentile_50_lower = np.percentile(all_predictions, 25, axis=0)
    
    # Create future dates
    last_date = dates.iloc[-1]
    future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_days, freq='B')
    
    # Calculate summary statistics
    current_price = dataset[-1]
    mean_final = mean_prediction[-1]
    median_final = median_prediction[-1]
    
    # Prepare lookback data for plotting
    lookback_dates = dates.iloc[-look_back:]
    lookback_data = dataset[-look_back:]
    
    # Print summary statistics
    if verbose:
        forecast_start = future_dates[0].strftime('%B %d, %Y (%A)')
        forecast_end = future_dates[-1].strftime('%B %d, %Y (%A)')
        
        print(f"\n{'='*60}")
        print(f"Monte Carlo Simulation Summary for {ticker}")
        print(f"{'='*60}")
        print(f"Number of Simulations: {n_simulations:,}")
        print(f"Forecast Horizon: {forecast_days} business days")
        print(f"Forecast Period: {forecast_start} → {forecast_end}")
        print(f"Look-back Period: {look_back} days")
        print(f"\nCurrent Price: ${current_price:.2f}")
        print(f"\nForecast (Day {forecast_days}):")
        print(f"  Mean Price: ${mean_final:.2f} ({(mean_final/current_price - 1)*100:+.2f}%)")
        print(f"  Median Price: ${median_final:.2f} ({(median_final/current_price - 1)*100:+.2f}%)")
        print(f"  95% CI: ${percentile_95_lower[-1]:.2f} - ${percentile_95_upper[-1]:.2f}")
        print(f"  80% CI: ${percentile_80_lower[-1]:.2f} - ${percentile_80_upper[-1]:.2f}")
        print(f"  50% CI: ${percentile_50_lower[-1]:.2f} - ${percentile_50_upper[-1]:.2f}")
        print(f"\nPrice Range:")
        print(f"  Minimum: ${np.min(all_predictions[:, -1]):.2f}")
        print(f"  Maximum: ${np.max(all_predictions[:, -1]):.2f}")
        print(f"  Std Dev: ${std_prediction[-1]:.2f}")
        print(f"{'='*60}\n")
    
    return {
        'ticker': ticker,
        'n_simulations': n_simulations,
        'forecast_days': forecast_days,
        'look_back': look_back,
        'predictions': all_predictions,
        'mean': mean_prediction,
        'median': median_prediction,
        'std': std_prediction,
        'ci_95': (percentile_95_lower, percentile_95_upper),
        'ci_80': (percentile_80_lower, percentile_80_upper),
        'ci_50': (percentile_50_lower, percentile_50_upper),
        'future_dates': future_dates,
        'current_price': current_price,
        'lookback_dates': lookback_dates,
        'lookback_data': lookback_data
    }

In [0]:
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates
from scipy.stats import gaussian_kde

In [0]:
def plot_forecast_with_ci(results, figsize=(14, 7)):
    """
    Plot historical data + forecast with confidence intervals
    
    Parameters:
    -----------
    results : dict
        Output from monte_carlo_simulation()
    figsize : tuple
        Figure size (width, height)
    """
    ticker = results['ticker']
    n_simulations = results['n_simulations']
    forecast_days = results['forecast_days']
    look_back = results['look_back']
    lookback_dates = results['lookback_dates']
    lookback_data = results['lookback_data']
    future_dates = results['future_dates']
    mean_prediction = results['mean']
    median_prediction = results['median']
    ci_95 = results['ci_95']
    ci_80 = results['ci_80']
    ci_50 = results['ci_50']
    current_price = results['current_price']
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Historical data
    ax.plot(lookback_dates, lookback_data, label=f'Historical Data (Last {look_back} days)', 
            linewidth=2.5, color='#2962ff', alpha=0.9)
    
    # Forecasts
    ax.plot(future_dates, mean_prediction, label='Mean Forecast', 
            linewidth=2.5, color='#9500ff', linestyle='-', marker='o', markersize=3)
    ax.plot(future_dates, median_prediction, label='Median Forecast', 
            linewidth=2, color='#ff8c00', linestyle='--', marker='s', markersize=3)
    
    # Confidence intervals
    ax.fill_between(future_dates, ci_95[0], ci_95[1], 
                    alpha=0.15, color='#9500ff', label='95% CI', edgecolor='#9500ff', linewidth=0.5)
    ax.fill_between(future_dates, ci_80[0], ci_80[1], 
                    alpha=0.25, color='#ff8c00', label='80% CI', edgecolor='#ff8c00', linewidth=0.5)
    ax.fill_between(future_dates, ci_50[0], ci_50[1], 
                    alpha=0.35, color='#b4e600', label='50% CI (IQR)', edgecolor='#b4e600', linewidth=0.5)
    
    # Forecast start marker
    ax.axvline(x=lookback_dates.iloc[-1], color='#666666', linestyle=':', 
              linewidth=2, alpha=0.7, label='Forecast Start')
    
    # Info box
    forecast_text = f"Forecast: {future_dates[0].strftime('%b %d')} - {future_dates[-1].strftime('%b %d, %Y')}\n({forecast_days} business days)"
    ax.text(0.02, 0.98, forecast_text, transform=ax.transAxes, 
            fontsize=10, verticalalignment='top', 
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='#e0e0e0', alpha=0.9))
    
    # Format dates
    ax.xaxis.set_major_formatter(DateFormatter('%b %d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    ax.set_title(f'{ticker} Stock Price - Monte Carlo Forecast\n{n_simulations:,} Simulations | {forecast_days}-Day Horizon | Current: ${current_price:.2f}', 
                pad=20)
    ax.set_xlabel('Date')
    ax.set_ylabel('Close Price ($)')
    ax.legend(loc='best', framealpha=0.95, fancybox=False, edgecolor='#e0e0e0')
    
    plt.tight_layout()
    plt.show()

In [0]:
def plot_simulation_paths(results, n_paths=200, figsize=(14, 7)):
    """
    Plot sample simulation paths with mean/median forecasts
    
    Parameters:
    -----------
    results : dict
        Output from monte_carlo_simulation()
    n_paths : int
        Number of simulation paths to display
    figsize : tuple
        Figure size (width, height)
    """
    ticker = results['ticker']
    n_simulations = results['n_simulations']
    predictions = results['predictions']
    future_dates = results['future_dates']
    mean_prediction = results['mean']
    median_prediction = results['median']
    ci_95 = results['ci_95']
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Sample paths
    sample_indices = np.random.choice(n_simulations, min(n_paths, n_simulations), replace=False)
    for idx in sample_indices:
        ax.plot(future_dates, predictions[idx, :], alpha=0.05, color='#666666', linewidth=0.8)
    
    # Key forecasts
    ax.plot(future_dates, mean_prediction, label='Mean Forecast', 
            linewidth=3, color='#9500ff', marker='o', markersize=4, zorder=5)
    ax.plot(future_dates, median_prediction, label='Median Forecast', 
            linewidth=2.5, color='#ff8c00', linestyle='--', marker='s', markersize=4, zorder=5)
    ax.plot(future_dates, ci_95[1], label='95% Upper Bound', 
            linewidth=1.5, color='#ff0059', linestyle=':', alpha=0.8)
    ax.plot(future_dates, ci_95[0], label='95% Lower Bound', 
            linewidth=1.5, color='#0fffdb', linestyle=':', alpha=0.8)
    
    # Format dates
    ax.xaxis.set_major_formatter(DateFormatter('%b %d\n%a'))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=5))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, ha='center', fontsize=8)
    
    ax.set_title(f'{ticker} - Simulation Trajectories (showing {min(n_paths, n_simulations)} paths)', pad=15)
    ax.set_xlabel('Forecast Date')
    ax.set_ylabel('Close Price ($)')
    ax.legend(loc='best', framealpha=0.95, fancybox=False, edgecolor='#e0e0e0', fontsize=9)
    
    plt.tight_layout()
    plt.show()

In [0]:
def plot_final_distribution(results, figsize=(10, 7)):
    """
    Plot distribution of final forecasted prices
    
    Parameters:
    -----------
    results : dict
        Output from monte_carlo_simulation()
    figsize : tuple
        Figure size (width, height)
    """
    ticker = results['ticker']
    forecast_days = results['forecast_days']
    predictions = results['predictions']
    current_price = results['current_price']
    mean_final = results['mean'][-1]
    median_final = results['median'][-1]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Histogram
    final_prices = predictions[:, -1]
    ax.hist(final_prices, bins=50, density=True, alpha=0.7, 
            color='#2962ff', edgecolor='#333333', linewidth=0.5)
    
    # KDE curve
    kde = gaussian_kde(final_prices)
    x_range = np.linspace(final_prices.min(), final_prices.max(), 200)
    ax.plot(x_range, kde(x_range), color='#ff0059', linewidth=2.5, label='Density Curve')
    
    # Reference lines
    ax.axvline(current_price, color='#333333', linestyle='--', linewidth=2, 
              label=f'Current: ${current_price:.2f}', alpha=0.8)
    ax.axvline(mean_final, color='#9500ff', linestyle='-', linewidth=2, 
              label=f'Mean: ${mean_final:.2f}', alpha=0.8)
    ax.axvline(median_final, color='#ff8c00', linestyle='--', linewidth=2, 
              label=f'Median: ${median_final:.2f}', alpha=0.8)
    
    ax.set_title(f'{ticker} - Distribution of {forecast_days}-Day Forecast', pad=15)
    ax.set_xlabel('Forecasted Price ($)')
    ax.set_ylabel('Density')
    ax.legend(loc='best', framealpha=0.95, fancybox=False, edgecolor='#e0e0e0')
    
    plt.tight_layout()
    plt.show()


In [0]:
def plot_forecast_evolution(results, figsize=(10, 7)):
    """
    Plot forecast uncertainty evolution over time
    
    Parameters:
    -----------
    results : dict
        Output from monte_carlo_simulation()
    figsize : tuple
        Figure size (width, height)
    """
    ticker = results['ticker']
    forecast_days = results['forecast_days']
    predictions = results['predictions']
    current_price = results['current_price']
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Calculate statistics by day
    days_range = np.arange(1, forecast_days + 1)
    mean_by_day = np.mean(predictions, axis=0)
    std_by_day = np.std(predictions, axis=0)
    
    # Plot mean with uncertainty bands
    ax.plot(days_range, mean_by_day, linewidth=2.5, color='#9500ff', 
            marker='o', markersize=3, label='Mean Forecast')
    ax.fill_between(days_range, mean_by_day - std_by_day, mean_by_day + std_by_day, 
                    alpha=0.3, color='#9500ff', label='±1 Std Dev')
    ax.fill_between(days_range, mean_by_day - 2*std_by_day, mean_by_day + 2*std_by_day, 
                    alpha=0.15, color='#ff8c00', label='±2 Std Dev')
    ax.axhline(y=current_price, color='#333333', linestyle='--', linewidth=2, 
              label=f'Current Price: ${current_price:.2f}', alpha=0.7)
    
    ax.set_title(f'{ticker} - Forecast Uncertainty Over Time', pad=15)
    ax.set_xlabel('Days Ahead')
    ax.set_ylabel('Price ($)')
    ax.legend(loc='best', framealpha=0.95, fancybox=False, edgecolor='#e0e0e0')
    
    plt.tight_layout()
    plt.show()


In [0]:
def plot_monte_carlo_dashboard(results):
    """
    Create complete dashboard with all 4 Monte Carlo plots
    
    Parameters:
    -----------
    results : dict
        Output from monte_carlo_simulation()
    """
    ticker = results['ticker']
    n_simulations = results['n_simulations']
    forecast_days = results['forecast_days']
    look_back = results['look_back']
    lookback_dates = results['lookback_dates']
    lookback_data = results['lookback_data']
    future_dates = results['future_dates']
    mean_prediction = results['mean']
    median_prediction = results['median']
    predictions = results['predictions']
    ci_95 = results['ci_95']
    ci_80 = results['ci_80']
    ci_50 = results['ci_50']
    current_price = results['current_price']
    mean_final = mean_prediction[-1]
    median_final = median_prediction[-1]
    
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    ax1 = fig.add_subplot(gs[0, :])
    ax2 = fig.add_subplot(gs[1, :])
    ax3 = fig.add_subplot(gs[2, 0])
    ax4 = fig.add_subplot(gs[2, 1])
    
    # Plot 1: Historical + Forecast with confidence intervals
    ax1.plot(lookback_dates, lookback_data, label=f'Historical Data (Last {look_back} days)', 
            linewidth=2.5, color='#2962ff', alpha=0.9)
    ax1.plot(future_dates, mean_prediction, label='Mean Forecast', 
            linewidth=2.5, color='#9500ff', linestyle='-', marker='o', markersize=3)
    ax1.plot(future_dates, median_prediction, label='Median Forecast', 
            linewidth=2, color='#ff8c00', linestyle='--', marker='s', markersize=3)
    
    ax1.fill_between(future_dates, ci_95[0], ci_95[1], 
                    alpha=0.15, color='#9500ff', label='95% CI', edgecolor='#9500ff', linewidth=0.5)
    ax1.fill_between(future_dates, ci_80[0], ci_80[1], 
                    alpha=0.25, color='#ff8c00', label='80% CI', edgecolor='#ff8c00', linewidth=0.5)
    ax1.fill_between(future_dates, ci_50[0], ci_50[1], 
                    alpha=0.35, color='#b4e600', label='50% CI (IQR)', edgecolor='#b4e600', linewidth=0.5)
    
    ax1.axvline(x=lookback_dates.iloc[-1], color='#666666', linestyle=':', 
              linewidth=2, alpha=0.7, label='Forecast Start')
    
    forecast_text = f"Forecast: {future_dates[0].strftime('%b %d')} - {future_dates[-1].strftime('%b %d, %Y')}\n({forecast_days} business days)"
    ax1.text(0.02, 0.98, forecast_text, transform=ax1.transAxes, 
            fontsize=10, verticalalignment='top', 
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='#e0e0e0', alpha=0.9))
    
    ax1.xaxis.set_major_formatter(DateFormatter('%b %d'))
    ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    ax1.set_title(f'{ticker} Stock Price - Monte Carlo Forecast\n{n_simulations:,} Simulations | {forecast_days}-Day Horizon | Current: ${current_price:.2f}', pad=20)
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Close Price ($)')
    ax1.legend(loc='best', framealpha=0.95, fancybox=False, edgecolor='#e0e0e0', fontsize=10)
    
    # Plot 2: Sample simulation paths
    sample_indices = np.random.choice(n_simulations, min(200, n_simulations), replace=False)
    for idx in sample_indices:
        ax2.plot(future_dates, predictions[idx, :], alpha=0.05, color='#666666', linewidth=0.8)
    
    ax2.plot(future_dates, mean_prediction, label='Mean Forecast', 
            linewidth=3, color='#9500ff', marker='o', markersize=4, zorder=5)
    ax2.plot(future_dates, median_prediction, label='Median Forecast', 
            linewidth=2.5, color='#ff8c00', linestyle='--', marker='s', markersize=4, zorder=5)
    ax2.plot(future_dates, ci_95[1], label='95% Upper Bound', 
            linewidth=1.5, color='#ff0059', linestyle=':', alpha=0.8)
    ax2.plot(future_dates, ci_95[0], label='95% Lower Bound', 
            linewidth=1.5, color='#0fffdb', linestyle=':', alpha=0.8)
    
    ax2.xaxis.set_major_formatter(DateFormatter('%b %d\n%a'))
    ax2.xaxis.set_major_locator(mdates.DayLocator(interval=5))
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=0, ha='center', fontsize=8)
    
    ax2.set_title(f'Simulation Trajectories (showing {min(200, n_simulations)} paths)', pad=15)
    ax2.set_xlabel('Forecast Date')
    ax2.set_ylabel('Close Price ($)')
    ax2.legend(loc='best', framealpha=0.95, fancybox=False, edgecolor='#e0e0e0', fontsize=9)
    
    # Plot 3: Distribution of final prices
    final_prices = predictions[:, -1]
    ax3.hist(final_prices, bins=50, density=True, alpha=0.7, 
            color='#2962ff', edgecolor='#333333', linewidth=0.5)
    
    kde = gaussian_kde(final_prices)
    x_range = np.linspace(final_prices.min(), final_prices.max(), 200)
    ax3.plot(x_range, kde(x_range), color='#ff0059', linewidth=2.5, label='Density Curve')
    
    ax3.axvline(current_price, color='#333333', linestyle='--', linewidth=2, 
              label=f'Current: ${current_price:.2f}', alpha=0.8)
    ax3.axvline(mean_final, color='#9500ff', linestyle='-', linewidth=2, 
              label=f'Mean: ${mean_final:.2f}', alpha=0.8)
    ax3.axvline(median_final, color='#ff8c00', linestyle='--', linewidth=2, 
              label=f'Median: ${median_final:.2f}', alpha=0.8)
    
    ax3.set_title(f'Distribution of {forecast_days}-Day Forecast', pad=10)
    ax3.set_xlabel('Forecasted Price ($)')
    ax3.set_ylabel('Density')
    ax3.legend(loc='best', framealpha=0.95, fancybox=False, edgecolor='#e0e0e0', fontsize=9)
    
    # Plot 4: Forecast evolution over time
    days_range = np.arange(1, forecast_days + 1)
    mean_by_day = np.mean(predictions, axis=0)
    std_by_day = np.std(predictions, axis=0)
    
    ax4.plot(days_range, mean_by_day, linewidth=2.5, color='#9500ff', 
            marker='o', markersize=3, label='Mean Forecast')
    ax4.fill_between(days_range, mean_by_day - std_by_day, mean_by_day + std_by_day, 
                    alpha=0.3, color='#9500ff', label='±1 Std Dev')
    ax4.fill_between(days_range, mean_by_day - 2*std_by_day, mean_by_day + 2*std_by_day, 
                    alpha=0.15, color='#ff8c00', label='±2 Std Dev')
    ax4.axhline(y=current_price, color='#333333', linestyle='--', linewidth=2, 
              label=f'Current Price: ${current_price:.2f}', alpha=0.7)
    
    ax4.set_title('Forecast Uncertainty Over Time', pad=10)
    ax4.set_xlabel('Days Ahead')
    ax4.set_ylabel('Price ($)')
    ax4.legend(loc='best', framealpha=0.95, fancybox=False, edgecolor='#e0e0e0', fontsize=9)
    
    plt.tight_layout()
    plt.show()

In [0]:
SIMULATIONS = 1_000
FORECAST = 20
ALPHA = 2
LOOKBACK = 260

In [0]:
apple_results = monte_carlo_simulation('AAPL', n_simulations=SIMULATIONS, forecast_days=FORECAST, look_back=LOOKBACK, alpha=ALPHA)

amazon_results = monte_carlo_simulation('AMZN', n_simulations=SIMULATIONS, forecast_days=FORECAST, look_back=LOOKBACK, alpha=ALPHA)

broadcom_results = monte_carlo_simulation('AVGO', n_simulations=SIMULATIONS, forecast_days=FORECAST, look_back=LOOKBACK, alpha=ALPHA)

google_results = monte_carlo_simulation('GOOGL', n_simulations=SIMULATIONS, forecast_days=FORECAST, look_back=LOOKBACK, alpha=ALPHA)

nvidia_results = monte_carlo_simulation('NVDA', n_simulations=SIMULATIONS, forecast_days=FORECAST, look_back=LOOKBACK, alpha=ALPHA)

microsoft_results = monte_carlo_simulation('MSFT', n_simulations=SIMULATIONS, forecast_days=FORECAST, look_back=LOOKBACK, alpha=ALPHA)

meta_results = monte_carlo_simulation('META', n_simulations=SIMULATIONS, forecast_days=FORECAST, look_back=LOOKBACK, alpha=ALPHA)

tesla_results = monte_carlo_simulation('TSLA', n_simulations=SIMULATIONS, forecast_days=FORECAST, look_back=LOOKBACK, alpha=ALPHA)

In [0]:
#plot_forecast_with_ci(tesla_results)          # Historical + Forecast with CI
#plot_simulation_paths(tesla_results)          # Sample simulation paths
#plot_final_distribution(tesla_results)        # Distribution of final prices
#plot_forecast_evolution(tesla_results)        # Uncertainty evolution
plot_monte_carlo_dashboard(tesla_results)

---

## Model Performance Evaluation

Comparing Monte Carlo forecasts against actual realized prices.

In [0]:
results_dict = {
    'AAPL': apple_results,
    'AMZN': amazon_results,
    'AVGO': broadcom_results,
    'GOOGL': google_results,
    'NVDA': nvidia_results,
    'MSFT': microsoft_results,
    'META': meta_results,
    'TSLA': tesla_results
}

# Actual prices 20 business days after forecast (March 13, 2026)
actual_prices_manual = {
    'META': 613.71,
    'AAPL': 250.12,
    'GOOGL': 301.46,
    'AMZN': 206.67,
    'AVGO': 322.16,
    'MSFT': 395.55,
    'NVDA': 180.25,
    'TSLA': 391.20
}

# Build evaluation dataframe from actual prices
actual_data = []
target_date_str = '2026-03-13'  # The forecast target date (20 business days after Feb 13)

for ticker, actual_price in actual_prices_manual.items():
    result = results_dict[ticker]
    
    # Get forecast values
    current_price = result['current_price']
    median_forecast = result['median'][-1]
    mean_forecast = result['mean'][-1]
    ci_95_lower = result['ci_95'][0][-1]
    ci_95_upper = result['ci_95'][1][-1]
    ci_80_lower = result['ci_80'][0][-1]
    ci_80_upper = result['ci_80'][1][-1]
    
    # Calculate errors
    median_error = actual_price - median_forecast
    median_error_pct = (median_error / current_price) * 100
    mean_error = actual_price - mean_forecast
    mean_error_pct = (mean_error / current_price) * 100
    
    # Calculate absolute percentage errors
    median_ape = abs((actual_price - median_forecast) / actual_price) * 100
    mean_ape = abs((actual_price - mean_forecast) / actual_price) * 100
    
    # Check if actual falls within confidence intervals
    in_95_ci = ci_95_lower <= actual_price <= ci_95_upper
    in_80_ci = ci_80_lower <= actual_price <= ci_80_upper
    
    # Actual change from current price
    actual_change_pct = ((actual_price - current_price) / current_price) * 100
    forecast_change_pct = ((median_forecast - current_price) / current_price) * 100
    
    actual_data.append({
        'Ticker': ticker,
        'Current Price': current_price,
        'Median Forecast': median_forecast,
        'Mean Forecast': mean_forecast,
        'Actual Price': actual_price,
        'Median Error ($)': median_error,
        'Median Error (%)': median_error_pct,
        'Mean Error ($)': mean_error,
        'Mean Error (%)': mean_error_pct,
        'Median APE (%)': median_ape,
        'Mean APE (%)': mean_ape,
        'Actual Change (%)': actual_change_pct,
        'Forecast Change (%)': forecast_change_pct,
        'In 95% CI': in_95_ci,
        'In 80% CI': in_80_ci,
        '95% CI Lower': ci_95_lower,
        '95% CI Upper': ci_95_upper,
        'Forecast Date': target_date_str
    })

# Create evaluation DataFrame
eval_df = pd.DataFrame(actual_data)
eval_df = eval_df.sort_values('Median APE (%)', ascending=True).reset_index(drop=True)

print(f"\nSuccessfully evaluated {len(eval_df)} stocks")
print(f"Forecast date: {target_date_str}")
print(f"Forecast start: 2026-02-13")
display(eval_df)

In [0]:
# Plot: Median Forecast vs Actual Price Comparison
fig, ax = plt.subplots(figsize=(14, 8))

# Prepare data for grouped bar chart
x = np.arange(len(eval_df))
width = 0.25

# Create grouped bars
bars1 = ax.bar(x - width, eval_df['Current Price'], width, 
              label='Current Price (Day 0)', color='#2962ff', alpha=0.8, edgecolor='#333333', linewidth=1)
bars2 = ax.bar(x, eval_df['Median Forecast'], width, 
              label='Median Forecast (Day 20)', color='#ff8c00', alpha=0.8, edgecolor='#333333', linewidth=1)
bars3 = ax.bar(x + width, eval_df['Actual Price'], width, 
              label='Actual Price (Day 20)', color='#b4e600', alpha=0.8, edgecolor='#333333', linewidth=1)

# Add value labels on actual price bars (with error)
for i, bar in enumerate(bars3):
    height = bar.get_height()
    error_pct = eval_df.iloc[i]['Median Error (%)']
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${height:.2f}\n({error_pct:+.1f}%)',
            ha='center', va='bottom',
            fontsize=9, color='#333333', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(eval_df['Ticker'], fontweight='bold')
ax.set_title(f'Forecast vs Actual Performance - {FORECAST} Day Horizon\nForecast Date: {eval_df.iloc[0]["Forecast Date"]}', pad=20)
ax.set_xlabel('Ticker')
ax.set_ylabel('Price ($)')
ax.legend(loc='best', framealpha=0.95, fancybox=False, edgecolor='#e0e0e0')

plt.tight_layout()
plt.show()

In [0]:
# Plot: Forecast Error Analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: Percentage errors
colors_error = ['#b4e600' if abs(x) < 5 else '#ff8c00' if abs(x) < 10 else '#ff0059' 
                for x in eval_df['Median Error (%)']]
bars = ax1.bar(eval_df['Ticker'], eval_df['Median Error (%)'], 
              color=colors_error, alpha=0.8, edgecolor='#333333', linewidth=1)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:+.2f}%',
            ha='center', va='bottom' if height > 0 else 'top',
            fontsize=10, fontweight='bold', color='#333333')

ax1.axhline(y=0, color='#666666', linestyle='-', linewidth=1.5, alpha=0.7)
ax1.set_title('Forecast Error (Median Forecast - Actual)', pad=15)
ax1.set_xlabel('Ticker')
ax1.set_ylabel('Error (%)')
ax1.grid(axis='y', alpha=0.3, linestyle='--')

# Right plot: Absolute Percentage Error (APE)
ax2.barh(eval_df['Ticker'], eval_df['Median APE (%)'], 
         color='#2962ff', alpha=0.8, edgecolor='#333333', linewidth=1)

# Add value labels
for i, (ticker, ape) in enumerate(zip(eval_df['Ticker'], eval_df['Median APE (%)'])):
    ax2.text(ape, i, f' {ape:.2f}%', 
            va='center', ha='left', fontsize=10, fontweight='bold', color='#333333')

ax2.set_title('Absolute Percentage Error (APE)', pad=15)
ax2.set_xlabel('APE (%)')
ax2.set_ylabel('Ticker')
ax2.grid(axis='x', alpha=0.3, linestyle='--')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

In [0]:
# Plot: Confidence Interval Coverage
fig, ax = plt.subplots(figsize=(14, 8))

# Sort by actual price for better visualization
df_sorted = eval_df.sort_values('Actual Price', ascending=True).reset_index(drop=True)
y_pos = np.arange(len(df_sorted))

# Plot confidence intervals
for idx, row in df_sorted.iterrows():
    # 95% CI
    ax.plot([row['95% CI Lower'], row['95% CI Upper']], [idx, idx], 
            linewidth=10, color='#9500ff', alpha=0.2, solid_capstyle='round', label='95% CI' if idx == 0 else '')
    
    # Median forecast
    ax.scatter(row['Median Forecast'], idx, s=120, color='#ff8c00', 
              marker='o', edgecolors='#333333', linewidths=2, zorder=6, 
              label='Median Forecast' if idx == 0 else '')
    
    # Actual price
    in_95_ci = row['In 95% CI']
    actual_color = '#b4e600' if in_95_ci else '#ff0059'
    marker_shape = 'D' if in_95_ci else 'X'
    ax.scatter(row['Actual Price'], idx, s=150, color=actual_color, 
              marker=marker_shape, edgecolors='#333333', linewidths=2, zorder=7, 
              label=f'Actual (In CI)' if idx == 0 and in_95_ci else f'Actual (Out of CI)' if idx == 0 and not in_95_ci else '')
    
    # Add APE text
    ax.text(row['95% CI Upper'] + 5, idx, f"APE: {row['Median APE (%)']:.1f}%", 
           va='center', ha='left', fontsize=9, color='#666666')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_sorted['Ticker'], fontsize=11, fontweight='bold')
ax.set_xlabel('Price ($)')
ax.set_ylabel('Ticker')
ax.set_title(f'Confidence Interval Coverage Analysis - {FORECAST} Day Forecast\n95% CI Coverage: {eval_df["In 95% CI"].sum()}/{len(eval_df)} stocks', pad=20)

# Custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#9500ff', alpha=0.2, label='95% Confidence Interval'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff8c00', 
              markersize=9, markeredgecolor='#333333', markeredgewidth=2, label='Median Forecast'),
    plt.Line2D([0], [0], marker='D', color='w', markerfacecolor='#b4e600', 
              markersize=9, markeredgecolor='#333333', markeredgewidth=2, label='Actual (Within CI)'),
    plt.Line2D([0], [0], marker='X', color='w', markerfacecolor='#ff0059', 
              markersize=9, markeredgecolor='#333333', markeredgewidth=2, label='Actual (Outside CI)')
]
ax.legend(handles=legend_elements, loc='best', framealpha=0.95, fancybox=False, edgecolor='#e0e0e0')

plt.tight_layout()
plt.show()

In [0]:
# Calculate comprehensive performance metrics
print("=" * 70)
print(f"MODEL PERFORMANCE EVALUATION - {FORECAST} DAY FORECAST")
print("=" * 70)
print(f"\nForecast Date: {eval_df.iloc[0]['Forecast Date']}")
print(f"Number of Stocks: {len(eval_df)}")

# Overall accuracy metrics
print("\n" + "-" * 70)
print("ACCURACY METRICS (Median Forecast):")
print("-" * 70)
mape = eval_df['Median APE (%)'].mean()
median_ape = eval_df['Median APE (%)'].median()
mae = abs(eval_df['Median Error ($)']).mean()
rmse = np.sqrt((eval_df['Median Error ($)']**2).mean())

print(f"\nMean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"Median Absolute Percentage Error:      {median_ape:.2f}%")
print(f"Mean Absolute Error (MAE):              ${mae:.2f}")
print(f"Root Mean Squared Error (RMSE):         ${rmse:.2f}")

# Bias analysis
print("\n" + "-" * 70)
print("BIAS ANALYSIS:")
print("-" * 70)
mean_bias = eval_df['Median Error ($)'].mean()
mean_bias_pct = eval_df['Median Error (%)'].mean()
overestimate_count = (eval_df['Median Error ($)'] < 0).sum()
underestimate_count = (eval_df['Median Error ($)'] > 0).sum()

print(f"\nMean Bias (Error):                      ${mean_bias:+.2f} ({mean_bias_pct:+.2f}%)")
if mean_bias < 0:
    print(f"Tendency:                               Overestimation")
elif mean_bias > 0:
    print(f"Tendency:                               Underestimation")
else:
    print(f"Tendency:                               Neutral")
print(f"\nOverestimated (Forecast > Actual):      {overestimate_count} stocks")
print(f"Underestimated (Forecast < Actual):     {underestimate_count} stocks")

# Confidence interval coverage
print("\n" + "-" * 70)
print("CONFIDENCE INTERVAL COVERAGE:")
print("-" * 70)
ci_95_coverage = (eval_df['In 95% CI'].sum() / len(eval_df)) * 100
ci_80_coverage = (eval_df['In 80% CI'].sum() / len(eval_df)) * 100

print(f"\n95% CI Coverage:                        {eval_df['In 95% CI'].sum()}/{len(eval_df)} stocks ({ci_95_coverage:.1f}%)")
print(f"80% CI Coverage:                        {eval_df['In 80% CI'].sum()}/{len(eval_df)} stocks ({ci_80_coverage:.1f}%)")
print(f"\nExpected 95% Coverage:                  ~95%")
print(f"Expected 80% Coverage:                  ~80%")

if ci_95_coverage >= 90:
    print(f"\n✓ 95% CI coverage is good (well-calibrated)")
else:
    print(f"\n✗ 95% CI coverage is below expected (under-calibrated)")

# Best and worst predictions
print("\n" + "-" * 70)
print("BEST PREDICTIONS (Lowest APE):")
print("-" * 70)
for idx, row in eval_df.head(3).iterrows():
    print(f"\n{idx + 1}. {row['Ticker']}")
    print(f"   Forecast: ${row['Median Forecast']:.2f} | Actual: ${row['Actual Price']:.2f}")
    print(f"   APE: {row['Median APE (%)']:.2f}% | Error: ${row['Median Error ($)']:+.2f}")
    print(f"   In 95% CI: {'Yes' if row['In 95% CI'] else 'No'}")

print("\n" + "-" * 70)
print("WORST PREDICTIONS (Highest APE):")
print("-" * 70)
for idx, row in eval_df.tail(3).iterrows():
    rank = len(eval_df) - idx
    print(f"\n{rank}. {row['Ticker']}")
    print(f"   Forecast: ${row['Median Forecast']:.2f} | Actual: ${row['Actual Price']:.2f}")
    print(f"   APE: {row['Median APE (%)']:.2f}% | Error: ${row['Median Error ($)']:+.2f}")
    print(f"   In 95% CI: {'Yes' if row['In 95% CI'] else 'No'}")

# Direction accuracy
print("\n" + "-" * 70)
print("DIRECTION ACCURACY:")
print("-" * 70)
actual_direction = eval_df['Actual Change (%)'] > 0
forecast_direction = eval_df['Forecast Change (%)'] > 0
direction_correct = (actual_direction == forecast_direction).sum()
direction_accuracy = (direction_correct / len(eval_df)) * 100

print(f"\nCorrect Direction:                      {direction_correct}/{len(eval_df)} stocks ({direction_accuracy:.1f}%)")
print(f"\nDirectional Breakdown:")
print(f"  Actual Up / Forecast Up:              {((actual_direction) & (forecast_direction)).sum()} stocks")
print(f"  Actual Down / Forecast Down:          {((~actual_direction) & (~forecast_direction)).sum()} stocks")
print(f"  Actual Up / Forecast Down:            {((actual_direction) & (~forecast_direction)).sum()} stocks (miss)")
print(f"  Actual Down / Forecast Up:            {((~actual_direction) & (forecast_direction)).sum()} stocks (miss)")

print("\n" + "=" * 70)